# RAG

## Business Setup 
* Insurance Tech Startup
* We have a knowledge base of the company shared drive - knowledge base folder
* task = AI Knowledge Worker

## 'Blunt Instrument' Implementation
* Read names of Employees/Products
* Identify if intent is employee or product related
* add relevant details to prompt


In [1]:
import os
import gradio as gr
from dotenv import load_dotenv
from pathlib import Path
import glob
from openai import OpenAI

In [2]:
load_dotenv(override = True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key.startswith('sk-'):
    print('API Key is valid')
else:
    print('API Key is invalid')




API Key is valid


In [4]:
MODEL = 'gpt-4.1-nano'
openai = OpenAI()

## Write Employee & Product Data in to a Dictionary

In [5]:
knowledge = {}
filenames = glob.glob("knowledge-base/employees/*") # * is a wildcard for all files

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, 'r', encoding='utf-8') as f:
        knowledge[name.lower()] = f.read()

In [6]:
knowledge["lancaster"]

"# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market t

In [7]:
filenames = glob.glob("knowledge-base/products/*")
for filename in filenames:
    name = Path(filename).stem
    with open(filename, 'r', encoding='utf-8') as f:
        knowledge[name.lower()] = f.read()

In [8]:
knowledge.keys()

dict_keys(['kim', 'patel', 'sharma', 'chen', 'spencer', 'foster', 'tran', 'blake', 'zhang', 'thompson', 'adams', 'liu', 'lancaster', 'park', 'greene', 'johnson', 'thomson', "o'brien", 'martinez', 'williams', 'rivera', 'trenton', 'anderson', 'harper', 'rodriguez', 'wilson', 'bishop', 'carter', 'brooks', 'walker', 'rellm', 'claimllm', 'bizllm', 'lifellm', 'healthllm', 'markellm', 'homellm', 'carllm'])

In [9]:
SYSTEM_PREFIX = """
You represent InsureLLM, the Insurance Tech Startup.
You are an expert in answering questions about InsureLLM; its employees and its products.
You are provided with additional context that might be relevant to the users question.
Gove brief accurate answers, if you don't know the answer say so.

Relevant context:

"""

def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace()) # keeps only alphabets and spaces (no punctuation)
    words = text.lower().split()
    relevant_context = []
    for word in words: # iterate through words
        if word in knowledge: # lookup in knowledge base
            relevant_context.append(knowledge[word]) # add to list
    return relevant_context

In [10]:
# more pyhtonic
def get_relevant_context(message):
    text=''.join(ch for ch in message in ch.isalpha() or ch.isspace())
    words=text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]